# mHC vs HC vs Baseline — Mini Colab T4 Training and Benchmark

Notebook này chạy **Mini controlled experiment** trên Google Colab T4 cho repo mHC: Manifold-Constrained Hyper-Connections.

Mục tiêu:
- Train from scratch ba biến thể: baseline Transformer residual, traditional HC, và mHC.
- Dùng cùng kiến trúc, cùng FineWeb10B shard budget, cùng optimizer, cùng effective batch, cùng số iteration.
- Báo cáo training loss, validation loss, validation perplexity, inference latency, throughput, và peak VRAM.
- Tạo checkpoint thật, validation metric thật, benchmark inference thật, bảng/biểu đồ sẵn đưa vào report.

**Honesty boundary:** Đây là controlled small-scale Colab T4 experiment, not full paper reproduction / không phải full paper-scale reproduction. Không claim reproduce MMLU/BBH/GSM8K hoặc quality benchmark của paper.


## 0. Runtime requirement

Chọn runtime GPU trong Colab:

`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

Nếu GPU không phải T4 vẫn chạy được, nhưng runtime/VRAM có thể khác report.


In [1]:
# Check GPU
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


Wed May 20 13:54:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive

Drive dùng để lưu checkpoints đã train, summaries, benchmark outputs, và figures. Training vẫn chạy local ở `/content/mhc` cho nhanh, sau đó notebook sync artifacts sang Drive.


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Global configuration — Mini is report preset

`EXPERIMENT_PRESET = "mini"` là workflow chính để nộp/report.

Preset:
- `tiny`: smoke/debug only, không dùng làm kết quả chính.
- `mini`: main report experiment, default.
- `mini_plus`: stronger but slower, optional nếu còn thời gian/GPU.


In [3]:
from pathlib import Path
import os, json, shutil, subprocess, time, textwrap, glob, csv, math

# TODO: set your repo URL before cloning in Colab.
REPO_URL = "https://github.com/khoaoe/mHC-manifold-constrained-hyper-connections.git"
BRANCH = "t4-inference-benchmark"  

DRIVE_ROOT = Path("/content/drive/MyDrive/mhc")
LOCAL_REPO = Path("/content/mhc")

EXPERIMENT_PRESET = "mini"  # "tiny", "mini", "mini_plus"

PRESETS = {
    "tiny": {
        "DATA_SHARDS": 1,
        "MAX_ITERS": 300,
        "EVAL_INTERVAL": 25,
        "EVAL_ITERS": 10,
        "BATCH_SIZE": 4,
        "GRAD_ACCUM": 4,
        "BLOCK_SIZE": 128,
        "N_LAYER": 2,
        "N_HEAD": 2,
        "N_EMBD": 128,
    },
    "mini": {
        "DATA_SHARDS": 1,
        "MAX_ITERS": 1500,
        "EVAL_INTERVAL": 75,
        "EVAL_ITERS": 20,
        "BATCH_SIZE": 2,
        "GRAD_ACCUM": 16,
        "BLOCK_SIZE": 256,
        "N_LAYER": 4,
        "N_HEAD": 4,
        "N_EMBD": 192,
    },
    "mini_plus": {
        "DATA_SHARDS": 3,
        "MAX_ITERS": 3000,
        "EVAL_INTERVAL": 100,
        "EVAL_ITERS": 30,
        "BATCH_SIZE": 2,
        "GRAD_ACCUM": 16,
        "BLOCK_SIZE": 256,
        "N_LAYER": 4,
        "N_HEAD": 4,
        "N_EMBD": 192,
    },
}

if EXPERIMENT_PRESET not in PRESETS:
    raise ValueError(f"unknown EXPERIMENT_PRESET={EXPERIMENT_PRESET!r}")

globals().update(PRESETS[EXPERIMENT_PRESET])

# Fixed runtime/training defaults.
DTYPE = "float16"
DEVICE = "cuda"
WANDB_LOG = "False"
COMPILE = "false"
COMPILE_MODEL = "False"
DATA_LOADER = "memmap"

# Inference benchmark defaults.
BENCH_BATCH_SIZE = 1
PROMPT_LEN = 128
GEN_LEN = 32
NUM_WARMUP = 5
NUM_ITERS = 20

RUN_NAME = f"t4-{EXPERIMENT_PRESET}-fineweb{DATA_SHARDS}shards-{MAX_ITERS}iters"
DRIVE_RUNS = DRIVE_ROOT / "runs" / RUN_NAME
DRIVE_REPORTS = DRIVE_ROOT / "reports" / RUN_NAME
LOG_DIR = DRIVE_REPORTS / "logs"

VARIANTS = {
    "baseline": {
        "config": "config/train_fineweb10B_mini_t4.py",
        "out_dir": "out-t4-mini-baseline",
        "expected": {"hc_disable": True, "mhc": False, "hc_num_streams": 1},
    },
    "hc": {
        "config": "config/train_fineweb10B_hc_mini_t4.py",
        "out_dir": "out-t4-mini-hc",
        "expected": {"hc_disable": False, "mhc": False, "hc_num_streams": 4},
    },
    "mhc": {
        "config": "config/train_fineweb10B_mhc_mini_t4.py",
        "out_dir": "out-t4-mini-mhc",
        "expected": {"hc_disable": False, "mhc": True, "hc_num_streams": 4},
    },
}

# Tiny preset uses tiny config files if selected, but mini remains report default.
if EXPERIMENT_PRESET == "tiny":
    VARIANTS["baseline"]["config"] = "config/train_fineweb10B_tiny_t4.py"
    VARIANTS["hc"]["config"] = "config/train_fineweb10B_hc_tiny_t4.py"
    VARIANTS["mhc"]["config"] = "config/train_fineweb10B_mhc_tiny_t4.py"
    VARIANTS["baseline"]["out_dir"] = "out-t4-tiny-baseline"
    VARIANTS["hc"]["out_dir"] = "out-t4-tiny-hc"
    VARIANTS["mhc"]["out_dir"] = "out-t4-tiny-mhc"
elif EXPERIMENT_PRESET == "mini_plus":
    VARIANTS["baseline"]["out_dir"] = "out-t4-mini-plus-baseline"
    VARIANTS["hc"]["out_dir"] = "out-t4-mini-plus-hc"
    VARIANTS["mhc"]["out_dir"] = "out-t4-mini-plus-mhc"

BASELINE_OUT = VARIANTS["baseline"]["out_dir"]
HC_OUT = VARIANTS["hc"]["out_dir"]
MHC_OUT = VARIANTS["mhc"]["out_dir"]
BASELINE_CONFIG = VARIANTS["baseline"]["config"]
HC_CONFIG = VARIANTS["hc"]["config"]
MHC_CONFIG = VARIANTS["mhc"]["config"]

print("selected preset:", EXPERIMENT_PRESET)
print("model shape:", f"L={N_LAYER}, H={N_HEAD}, D={N_EMBD}, block={BLOCK_SIZE}")
print("effective batch tokens/iter:", BATCH_SIZE * GRAD_ACCUM * BLOCK_SIZE)
print("dataset train shards:", DATA_SHARDS)
print("RUN_NAME:", RUN_NAME)
print("DRIVE_RUNS:", DRIVE_RUNS)
print("DRIVE_REPORTS:", DRIVE_REPORTS)
print("local out dirs:")
for name, spec in VARIANTS.items():
    print(f"  {name}: examples/nanogpt/{spec['out_dir']}")


selected preset: mini
model shape: L=4, H=4, D=192, block=256
effective batch tokens/iter: 8192
dataset train shards: 1
RUN_NAME: t4-mini-fineweb1shards-1500iters
DRIVE_RUNS: /content/drive/MyDrive/mhc/runs/t4-mini-fineweb1shards-1500iters
DRIVE_REPORTS: /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters
local out dirs:
  baseline: examples/nanogpt/out-t4-mini-baseline
  hc: examples/nanogpt/out-t4-mini-hc
  mhc: examples/nanogpt/out-t4-mini-mhc


## 3. Clone repo and install dependencies

Nếu `REPO_URL` còn placeholder, sửa cell config trước. Nếu repo đã tồn tại trong `/content/mhc`, cell này pull branch hiện tại.


In [4]:
%cd /content

if str(REPO_URL).startswith("https://github.com/<"):
    raise ValueError("Set REPO_URL to your GitHub repo first.")

if LOCAL_REPO.exists():
    %cd /content/mhc
    !git fetch origin {BRANCH} --prune
    !git checkout {BRANCH}

    # Older notebook compatibility cell may have generated these files locally.
    # Remote branch now tracks them, so untracked local copies must be moved before pull.
    from pathlib import Path
    import shutil, subprocess, time

    generated_config_paths = [
        "examples/nanogpt/config/train_fineweb10B_hc_mini_t4.py",
        "examples/nanogpt/config/train_fineweb10B_hc_tiny_t4.py",
        "examples/nanogpt/config/train_fineweb10B_mhc_mini_t4.py",
        "examples/nanogpt/config/train_fineweb10B_mhc_tiny_t4.py",
        "examples/nanogpt/config/train_fineweb10B_mini_t4.py",
        "examples/nanogpt/config/train_fineweb10B_tiny_t4.py",
    ]
    backup_dir = Path("/content/mhc_untracked_backup") / time.strftime("%Y%m%d-%H%M%S")

    for rel in generated_config_paths:
        path = Path(rel)
        if not path.exists():
            continue
        proc = subprocess.run(
            ["git", "ls-files", "--others", "--exclude-standard", "--", rel],
            text=True,
            stdout=subprocess.PIPE,
            check=True,
        )
        if proc.stdout.strip() == rel:
            target = backup_dir / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(path), str(target))
            print(f"moved untracked generated file to backup: {rel} -> {target}")

    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} /content/mhc
    %cd /content/mhc
    !git pull --ff-only origin {BRANCH}

!git status --short
!git rev-parse --short HEAD


/content
/content/mhc
From https://github.com/khoaoe/mHC-manifold-constrained-hyper-connections
 * branch            t4-inference-benchmark -> FETCH_HEAD
Already on 't4-inference-benchmark'
Your branch is up to date with 'origin/t4-inference-benchmark'.
From https://github.com/khoaoe/mHC-manifold-constrained-hyper-connections
 * branch            t4-inference-benchmark -> FETCH_HEAD
Already up to date.
a7dec54


In [5]:
%cd /content/mhc

# Install project + example dependencies. tqdm/matplotlib needed for progress/figures.
!python -m pip install -q -e ".[examples]"
!python -m pip install -q tqdm matplotlib pandas

# Fast sanity checks that do not train. Some submitted/exported repos may not include tests; skip missing tests instead of failing notebook.
from pathlib import Path
import subprocess, sys, textwrap

candidate_tests = [
    "tests/test_benchmark_summary.py",
    "tests/test_training_summary.py",
    "tests/test_t4_tiny_workflow.py",
]
existing_tests = [t for t in candidate_tests if Path(t).exists()]
missing_tests = [t for t in candidate_tests if not Path(t).exists()]
if missing_tests:
    print("[WARN] missing optional test files; skipping:", missing_tests)
if existing_tests:
    subprocess.run([sys.executable, "-m", "pytest", "-q", *existing_tests], check=True)
else:
    print("[WARN] no optional pytest files found; continuing")

# Colab compatibility: ensure Mini/Tiny configs exist even if GitHub branch is older than this notebook.
config_dir = Path("examples/nanogpt/config")
config_dir.mkdir(parents=True, exist_ok=True)

def write_config(path, lines):
    path = config_dir / path
    if path.exists():
        print("exists:", path)
        return
    path.write_text("\n".join(lines).strip() + "\n", encoding="utf-8")
    print("created:", path)

COMMON_TINY = [
    'dataset = "fineweb10B"',
    'data_loader = "memmap"',
    'block_size = 128',
    'n_layer = 2',
    'n_head = 2',
    'n_embd = 128',
    'dropout = 0.0',
    'bias = False',
    'batch_size = 4',
    'gradient_accumulation_steps = 8',
    'max_iters = 1000',
    'eval_interval = 50',
    'log_interval = 5',
    'eval_iters = 20',
    'learning_rate = 6e-4',
    'weight_decay = 0.1',
    'beta1 = 0.9',
    'beta2 = 0.95',
    'grad_clip = 1.0',
    'warmup_iters = 50',
    'lr_decay_iters = 1000',
    'min_lr = 6e-5',
    'dtype = "float16"',
    'compile_model = False',
    'wandb_log = False',
]
COMMON_MINI = [
    'dataset = "fineweb10B"',
    'data_loader = "memmap"',
    'block_size = 256',
    'n_layer = 4',
    'n_head = 4',
    'n_embd = 192',
    'dropout = 0.0',
    'bias = False',
    'batch_size = 2',
    'gradient_accumulation_steps = 16',
    'max_iters = 1500',
    'eval_interval = 75',
    'log_interval = 5',
    'eval_iters = 20',
    'learning_rate = 6e-4',
    'weight_decay = 0.1',
    'beta1 = 0.9',
    'beta2 = 0.95',
    'grad_clip = 1.0',
    'warmup_iters = 75',
    'lr_decay_iters = 1500',
    'min_lr = 6e-5',
    'dtype = "float16"',
    'compile_model = False',
    'wandb_log = False',
]
BASELINE = [
    'hc_num_streams = 1',
    'hc_num_fracs = 1',
    'hc_disable = True',
    'mhc = False',
]
HC = [
    'hc_num_streams = 4',
    'hc_num_fracs = 1',
    'hc_disable = False',
    'mhc = False',
]
MHC = [
    'hc_num_streams = 4',
    'hc_num_fracs = 1',
    'hc_disable = False',
    'mhc = True',
    'sinkhorn_iters = 10',
    'sinkhorn_tau = 0.05',
    'mhc_h_res_proj = "sinkhorn"',
    'ns_steps = 5',
    'ns_eps = 1e-7',
    'ns_coeffs = (3.0, -3.2, 1.2)',
    'mhc_residual_identity_mix = False',
    'mhc_residual_alpha = 0.01',
]
write_config("train_fineweb10B_tiny_t4.py", ['out_dir = "out-t4-tiny-baseline"', 'wandb_run_name = "t4-tiny-baseline"'] + COMMON_TINY + BASELINE)
write_config("train_fineweb10B_hc_tiny_t4.py", ['out_dir = "out-t4-tiny-hc"', 'wandb_run_name = "t4-tiny-hc"'] + COMMON_TINY + HC)
write_config("train_fineweb10B_mhc_tiny_t4.py", ['out_dir = "out-t4-tiny-mhc"', 'wandb_run_name = "t4-tiny-mhc"'] + COMMON_TINY + MHC)
write_config("train_fineweb10B_mini_t4.py", ['out_dir = "out-t4-mini-baseline"', 'wandb_run_name = "t4-mini-baseline"'] + COMMON_MINI + BASELINE)
write_config("train_fineweb10B_hc_mini_t4.py", ['out_dir = "out-t4-mini-hc"', 'wandb_run_name = "t4-mini-hc"'] + COMMON_MINI + HC)
write_config("train_fineweb10B_mhc_mini_t4.py", ['out_dir = "out-t4-mini-mhc"', 'wandb_run_name = "t4-mini-mhc"'] + COMMON_MINI + MHC)


/content/mhc
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mhc-hyper-connections (pyproject.toml) ... done
exists: examples/nanogpt/config/train_fineweb10B_tiny_t4.py
exists: examples/nanogpt/config/train_fineweb10B_hc_tiny_t4.py
exists: examples/nanogpt/config/train_fineweb10B_mhc_tiny_t4.py
exists: examples/nanogpt/config/train_fineweb10B_mini_t4.py
exists: examples/nanogpt/config/train_fineweb10B_hc_mini_t4.py
exists: examples/nanogpt/config/train_fineweb10B_mhc_mini_t4.py


## 4. Download FineWeb10B GPT-2 shards

Default Mini uses `DATA_SHARDS = 1` train shard plus validation shard. This keeps Colab T4 disk/RAM pressure low.

Training uses `data_loader="memmap"`, so extra shards should not be concatenated into RAM. If you switch to eager loading, old extra shards can increase RAM usage.


In [6]:
%cd /content/mhc

!python examples/nanogpt/data/fineweb10B/download.py {DATA_SHARDS}

!free -h
!du -sh examples/nanogpt/data/fineweb10B
!ls -lh examples/nanogpt/data/fineweb10B/*.bin | head


/content/mhc
  - 1 validation shard
  - 1 training shards

  fineweb_val_000000.bin already exists, skipping
  fineweb_train_000001.bin already exists, skipping

Done!
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       2.2Gi       2.0Mi       9.3Gi        11Gi
Swap:             0B          0B          0B
382M	examples/nanogpt/data/fineweb10B
-rw-r--r-- 1 root root 191M May 20 12:37 examples/nanogpt/data/fineweb10B/fineweb_train_000001.bin
-rw-r--r-- 1 root root 191M May 20 12:37 examples/nanogpt/data/fineweb10B/fineweb_val_000000.bin


### Optional cleanup: reset FineWeb train shards

Default now removes old extra train shards and keeps exactly `DATA_SHARDS` train files. This prevents older eager loaders from loading many stale shards into RAM. Set `RESET_TRAIN_SHARDS = False` only if you intentionally want to keep extra shards.


In [7]:
# Optional cleanup; disabled by default.
RESET_TRAIN_SHARDS = True

if RESET_TRAIN_SHARDS:
    data_dir = Path("/content/mhc/examples/nanogpt/data/fineweb10B")
    keep = {f"fineweb_train_{i:06d}.bin" for i in range(1, DATA_SHARDS + 1)}
    for path in data_dir.glob("fineweb_train_*.bin"):
        if path.name not in keep:
            print("removing", path)
            path.unlink()
    !ls -lh /content/mhc/examples/nanogpt/data/fineweb10B/*.bin | head
else:
    print("RESET_TRAIN_SHARDS=False; no files removed")


-rw-r--r-- 1 root root 191M May 20 12:37 /content/mhc/examples/nanogpt/data/fineweb10B/fineweb_train_000001.bin
-rw-r--r-- 1 root root 191M May 20 12:37 /content/mhc/examples/nanogpt/data/fineweb10B/fineweb_val_000000.bin


## 5. Smoke test — not for reporting

This cell only checks environment, data, model construction, memmap loader, checkpoint write, and summary write. Do **not** report these numbers.

Smoke values: `max_iters=20`, `eval_interval=5`, `eval_iters=2`, `batch_size=1`, `gradient_accumulation_steps=1`, `block_size=128`, `n_layer=2`, `n_head=2`, `n_embd=128`.


In [8]:
%cd /content/mhc/examples/nanogpt

SMOKE_OUT = "out-smoke-tiny-baseline"

!python -u train.py config/train_fineweb10B_tiny_t4.py \
  "out_dir='{SMOKE_OUT}'" \
  max_iters=20 eval_interval=5 eval_iters=2 log_interval=1 \
  batch_size=1 gradient_accumulation_steps=1 \
  block_size=128 n_layer=2 n_head=2 n_embd=128 \
  "device='{DEVICE}'" "dtype='{DTYPE}'" \
  wandb_log=False compile_model=False data_loader='memmap' \
  use_tqdm=True heartbeat_interval_s=60

!ls -lh {SMOKE_OUT}
!cat {SMOKE_OUT}/summary.json


/content/mhc/examples/nanogpt
Found 1 train shards, 1 val shards
Data loader: memmap
Train tokens: 100,000,000, Val tokens: 100,000,000
Training on cuda, dtype=float16, DDP=False
  tokens per iteration: 128
  model params: 6,849,152

iter 0: train loss 10.8560, val loss 10.8689
iter 0: loss 10.8661, lr 1.18e-05, time 113ms, tok/s 1132
iter 1: loss 10.9047, lr 2.35e-05, time 7ms, tok/s 18145                                                                                          
iter 2: loss 10.8535, lr 3.53e-05, time 6ms, tok/s 22558                                                                                                  
iter 3: loss 10.8256, lr 4.71e-05, time 5ms, tok/s 24421                                                                                                 
iter 4: loss 10.8257, lr 5.88e-05, time 5ms, tok/s 24321                                                                                                 
iter 5: train loss 10.8479, val loss 10.8351         

## 6. Helpers: train one variant, sync to Drive, verify outputs

Training logs stream live in Colab via `python -u train.py`. Each variant writes local logs and Drive logs. After each variant finishes, checkpoint/report artifacts sync to Drive and `verify_variant()` runs immediately.


In [9]:
import subprocess, shutil, os, json, time
from pathlib import Path
from datetime import datetime

REPO = Path("/content/mhc")
NANOGPT = REPO / "examples" / "nanogpt"
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
DRIVE_REPORTS.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SCRIPT = NANOGPT / "train.py"
SUMMARY_SCRIPT = NANOGPT / "summarize_training_runs.py"
BENCHMARK_SCRIPT = NANOGPT / "benchmark_inference.py"
PLOT_SCRIPT = NANOGPT / "plot_training_comparison.py"
RUN_ARTIFACTS = [
    "ckpt.pt",
    "summary.json",
    "config_effective.json",
    "dataset_manifest.json",
    "metrics.jsonl",
]



def _find_repo_file(filename, *, root=REPO):
    """Find helper file in repo when notebook runs against older/slightly different layout."""
    root = Path(root)
    preferred = [
        root / "examples" / "nanogpt" / filename,
        root / filename,
    ]
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(root.rglob(filename)) if root.exists() else []
    return matches[0] if matches else None


def _resolve_repo_file(path, *, filename=None, required=True):
    path = Path(path)
    if path.exists():
        return path
    found = _find_repo_file(filename or path.name)
    if found is not None:
        return found
    if required:
        raise FileNotFoundError(f"required file missing: {path}")
    return path


TRAIN_SCRIPT = _resolve_repo_file(TRAIN_SCRIPT, filename="train.py")
SUMMARY_SCRIPT = _resolve_repo_file(SUMMARY_SCRIPT, filename="summarize_training_runs.py", required=False)
BENCHMARK_SCRIPT = _resolve_repo_file(BENCHMARK_SCRIPT, filename="benchmark_inference.py", required=False)
PLOT_SCRIPT = _resolve_repo_file(PLOT_SCRIPT, filename="plot_training_comparison.py", required=False)


def run_cmd_live(cmd, *, cwd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("command:", " ".join(str(x) for x in cmd))
    print("log:", log_path)
    with log_path.open("a", encoding="utf-8") as log:
        log.write(f"\n===== start {datetime.now().isoformat(timespec='seconds')} =====\n")
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        code = proc.wait()
        log.write(f"===== end {datetime.now().isoformat(timespec='seconds')} code={code} =====\n")
    if code != 0:
        try:
            tail = "".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-80:])
        except Exception as exc:
            tail = f"<could not read log tail: {exc}>"
        raise RuntimeError(
            f"command failed with exit code {code}: {' '.join(str(x) for x in cmd)}\n"
            f"log path: {log_path}\n"
            f"last log lines:\n{tail}"
        )


def preflight_repo_files():
    required = [
        TRAIN_SCRIPT,
        NANOGPT / BASELINE_CONFIG,
        NANOGPT / HC_CONFIG,
        NANOGPT / MHC_CONFIG,
    ]
    missing = [str(path) for path in required if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(
            "required train/config files missing. Re-run install/dependencies compatibility cell first. Missing: "
            + ", ".join(missing)
        )
    print("[OK] preflight train/config files exist")


def sync_variant_to_drive(name):
    out_dir = NANOGPT / VARIANTS[name]["out_dir"]
    drive_dir = DRIVE_RUNS / VARIANTS[name]["out_dir"]
    if drive_dir.exists():
        shutil.rmtree(drive_dir)
    shutil.copytree(out_dir, drive_dir)
    print(f"synced {out_dir} -> {drive_dir}")


def _missing_run_artifacts(run_dir):
    return [item for item in RUN_ARTIFACTS if not (Path(run_dir) / item).exists()]


def _backup_incomplete_run(path, *, reason):
    path = Path(path)
    if not path.exists():
        return
    backup_root = Path("/content/mhc_incomplete_runs") / time.strftime("%Y%m%d-%H%M%S")
    target = backup_root / path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(path), str(target))
    print(f"moved incomplete run ({reason}) to backup: {path} -> {target}")


def restore_variant_from_drive(name):
    out_dir = NANOGPT / VARIANTS[name]["out_dir"]
    drive_dir = DRIVE_RUNS / VARIANTS[name]["out_dir"]

    if out_dir.exists():
        missing = _missing_run_artifacts(out_dir)
        if not missing:
            return True
        if (out_dir / "ckpt.pt").exists():
            print(f"[WARN] local {name} run incomplete, missing {missing}; will retrain")
            _backup_incomplete_run(out_dir, reason=f"local missing {missing}")

    if drive_dir.exists():
        missing = _missing_run_artifacts(drive_dir)
        if not missing:
            if out_dir.exists():
                shutil.rmtree(out_dir)
            shutil.copytree(drive_dir, out_dir)
            print(f"restored complete run {drive_dir} -> {out_dir}")
            return True
        if (drive_dir / "ckpt.pt").exists():
            print(f"[WARN] drive {name} run incomplete, missing {missing}; ignoring cached checkpoint and retraining")

    return False


def verify_variant(name):
    spec = VARIANTS[name]
    out_dir = NANOGPT / spec["out_dir"]
    missing = _missing_run_artifacts(out_dir)
    if missing:
        raise FileNotFoundError(f"{name} missing required artifacts: {missing}")
    summary = json.loads((out_dir / "summary.json").read_text())
    cfg = json.loads((out_dir / "config_effective.json").read_text())
    if summary.get("ok") is not True:
        raise RuntimeError(f"{name} summary ok is not True: {summary.get('error')}")
    expected_budget = {
        "max_iters": MAX_ITERS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps_total": GRAD_ACCUM,
        "n_layer": N_LAYER,
        "n_head": N_HEAD,
        "n_embd": N_EMBD,
        "block_size": BLOCK_SIZE,
    }
    for key, expected in expected_budget.items():
        actual = cfg.get(key)
        if actual != expected:
            raise AssertionError(f"{name} config mismatch {key}: {actual} != {expected}")
    for key, expected in spec["expected"].items():
        actual = cfg.get(key)
        if actual != expected:
            raise AssertionError(f"{name} residual flag mismatch {key}: {actual} != {expected}")
    print(f"[OK] {name}: best_val_loss={summary.get('best_val_loss')} iter_num={summary.get('iter_num')}")
    return True


def train_variant(name, *, force=False):
    preflight_repo_files()
    spec = VARIANTS[name]
    out_dir = NANOGPT / spec["out_dir"]
    log_path = LOG_DIR / f"{name}.log"

    if not force and restore_variant_from_drive(name):
        print(f"[SKIP] complete run exists for {name}: {out_dir}")
        verify_variant(name)
        return

    if out_dir.exists():
        if force:
            shutil.rmtree(out_dir)
        else:
            missing = _missing_run_artifacts(out_dir)
            if missing:
                _backup_incomplete_run(out_dir, reason=f"pre-train missing {missing}")


    cmd = [
        "python", "-u", str(TRAIN_SCRIPT), spec["config"],
        f"out_dir='{spec['out_dir']}'",
        f"max_iters={MAX_ITERS}",
        f"eval_interval={EVAL_INTERVAL}",
        f"eval_iters={EVAL_ITERS}",
        "log_interval=5",
        f"batch_size={BATCH_SIZE}",
        f"gradient_accumulation_steps={GRAD_ACCUM}",
        f"block_size={BLOCK_SIZE}",
        f"n_layer={N_LAYER}",
        f"n_head={N_HEAD}",
        f"n_embd={N_EMBD}",
        f"device='{DEVICE}'",
        f"dtype='{DTYPE}'",
        f"wandb_log={WANDB_LOG}",
        "compile_model=False",
        f"data_loader='{DATA_LOADER}'",
        "use_tqdm=True",
        "heartbeat_interval_s=60",
    ]

    start = time.time()
    print(f"\n===== TRAIN {name} start {datetime.now().isoformat(timespec='seconds')} =====")
    run_cmd_live(cmd, cwd=NANOGPT, log_path=log_path)
    elapsed_min = (time.time() - start) / 60
    print(f"===== TRAIN {name} end {datetime.now().isoformat(timespec='seconds')} elapsed_min={elapsed_min:.1f} =====")
    verify_variant(name)
    sync_variant_to_drive(name)

preflight_repo_files()


[OK] preflight train/config files exist


## 7. Timeout / stop advice

If one Mini variant takes too long:

1. Stop runtime.
2. Set `MAX_ITERS = 500` or `1000` in preset/config cell.
3. Re-run all three variants with exactly same new budget.

Do **not** report mixed budgets like baseline=1500 and mHC=500. Fair comparison requires same architecture, same dataset shard count, same optimizer, same effective batch, same `max_iters`, same eval settings.


## 8. Main Mini Experiment: Train baseline, HC, and mHC

This is the main result to report.

All variants use same architecture/training budget. Only residual connection mechanism changes:
- baseline: standard Transformer residual
- HC: Hyper-Connections
- mHC: manifold-constrained Hyper-Connections


In [10]:
%cd /content/mhc

for variant in ["baseline", "hc", "mhc"]:
    train_variant(variant, force=False)
    verify_variant(variant)
    

/content/mhc
[OK] preflight train/config files exist
[WARN] local baseline run incomplete, missing ['metrics.jsonl']; will retrain
moved incomplete run (local missing ['metrics.jsonl']) to backup: /content/mhc/examples/nanogpt/out-t4-mini-baseline -> /content/mhc_incomplete_runs/20260520-135533/out-t4-mini-baseline

===== TRAIN baseline start 2026-05-20T13:55:33 =====
command: python -u /content/mhc/examples/nanogpt/train.py config/train_fineweb10B_mini_t4.py out_dir='out-t4-mini-baseline' max_iters=1500 eval_interval=75 eval_iters=20 log_interval=5 batch_size=2 gradient_accumulation_steps=16 block_size=256 n_layer=4 n_head=4 n_embd=192 device='cuda' dtype='float16' wandb_log=False compile_model=False data_loader='memmap' use_tqdm=True heartbeat_interval_s=60
log: /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/logs/baseline.log
Found 1 train shards, 1 val shards
Data loader: memmap
Train tokens: 100,000,000, Val tokens: 100,000,000
Training on cuda, dtype=float16, 

## 9. Summarize training runs

Creates:
- `training_summary.csv`
- `training_summary.md`
- `training_summary.json`

These include real validation loss, perplexity, checkpoint status, config, and run metadata.


In [11]:
%cd /content/mhc

DRIVE_REPORTS.mkdir(parents=True, exist_ok=True)

!python examples/nanogpt/summarize_training_runs.py \
  --runs \
  "baseline=examples/nanogpt/{BASELINE_OUT}" \
  "hc=examples/nanogpt/{HC_OUT}" \
  "mhc=examples/nanogpt/{MHC_OUT}" \
  --output-dir "{DRIVE_REPORTS}"

!ls -lh "{DRIVE_REPORTS}"
!cat "{DRIVE_REPORTS}/training_summary.md"


/content/mhc
wrote /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/training_summary.csv
wrote /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/training_summary.md
wrote /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/training_summary.json
total 14K
drwx------ 2 root root 4.0K May 20 14:37 logs
-rw------- 1 root root 2.5K May 20 14:37 training_summary.csv
-rw------- 1 root root 4.5K May 20 14:37 training_summary.json
-rw------- 1 root root 2.8K May 20 14:37 training_summary.md
# Training Summary

| variant | run_dir | checkpoint_path | checkpoint_exists | ok | status | best_val_loss | final_train_loss | final_val_loss | final_val_ppl | tokens_seen | iter_num | elapsed_s | device_type | dtype | dataset | max_iters | eval_interval | batch_size | gradient_accumulation_steps | n_layer | n_head | n_embd | block_size | hc_num_streams | hc_disable | mhc | command |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---

## 10. Benchmark inference

Benchmark uses trained Mini checkpoints and synthetic token IDs. It measures runtime, not language quality:
- prefill latency
- full-context decode latency
- tokens/sec
- peak VRAM


In [12]:
%cd /content/mhc

if not BENCHMARK_SCRIPT.exists():
    found = _find_repo_file("benchmark_inference.py")
    if found is not None:
        BENCHMARK_SCRIPT = found
    else:
        raise FileNotFoundError(
            "benchmark_inference.py missing. Pull latest repo or copy examples/nanogpt/benchmark_inference.py."
        )

summarize_benchmarks = NANOGPT / "summarize_benchmarks.py"
if not summarize_benchmarks.exists():
    found = _find_repo_file("summarize_benchmarks.py")
    if found is not None:
        summarize_benchmarks = found
    else:
        raise FileNotFoundError("summarize_benchmarks.py missing. Pull latest repo or copy helper script.")

BENCH_DIR = DRIVE_REPORTS / "benchmarks"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

BENCH_SPECS = {
    "baseline": (f"examples/nanogpt/{BASELINE_OUT}/ckpt.pt", f"examples/nanogpt/{BASELINE_CONFIG}"),
    "hc": (f"examples/nanogpt/{HC_OUT}/ckpt.pt", f"examples/nanogpt/{HC_CONFIG}"),
    "mhc": (f"examples/nanogpt/{MHC_OUT}/ckpt.pt", f"examples/nanogpt/{MHC_CONFIG}"),
}

for name, (ckpt, cfg) in BENCH_SPECS.items():
    if not Path(ckpt).exists():
        raise FileNotFoundError(f"missing checkpoint for benchmark: {ckpt}")
    run_cmd_live(
        [
            "python", "-u", str(BENCHMARK_SCRIPT),
            "--ckpt", ckpt,
            "--config", cfg,
            "--device", DEVICE,
            "--dtype", DTYPE,
            "--batch-size", str(BENCH_BATCH_SIZE),
            "--prompt-len", str(PROMPT_LEN),
            "--gen-len", str(GEN_LEN),
            "--num-warmup", str(NUM_WARMUP),
            "--num-iters", str(NUM_ITERS),
            "--compile", COMPILE,
            "--output-json", str(BENCH_DIR / f"{name}.json"),
            "--output-csv", str(BENCH_DIR / f"{name}.csv"),
        ],
        cwd=REPO,
        log_path=LOG_DIR / f"benchmark-{name}.log",
    )

subprocess.run(["python", str(summarize_benchmarks), str(BENCH_DIR)], cwd=str(REPO), check=True)
print((BENCH_DIR / "summary.md").read_text())


/content/mhc
command: python -u /content/mhc/examples/nanogpt/benchmark_inference.py --ckpt examples/nanogpt/out-t4-mini-baseline/ckpt.pt --config examples/nanogpt/config/train_fineweb10B_mini_t4.py --device cuda --dtype float16 --batch-size 1 --prompt-len 128 --gen-len 32 --num-warmup 5 --num-iters 20 --compile false --output-json /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/benchmarks/baseline.json --output-csv /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/benchmarks/baseline.csv
log: /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/logs/benchmark-baseline.log
Benchmark summary:
device: cuda
dtype: float16
parameter_count: 11478720
prefill_latency_ms.mean: 3.405
full_context_generation_latency_ms.mean: 103.753
full_context_generation_latency_per_token_ms.mean: 3.242
tokens_per_sec.mean: 317.339
peak_vram: 61.48 MB
command: python -u /content/mhc/examples/nanogpt/benchmark_inference.py --ckpt examples/nanogpt/out-t4-mini-

## 11. Text generation demo

This is only a demo generation from the trained mHC Mini checkpoint. Model quality depends on small training budget, so do not overclaim text quality.


In [13]:
%cd /content/mhc

!python examples/nanogpt/infer.py \
  --ckpt examples/nanogpt/{MHC_OUT}/ckpt.pt \
  --config examples/nanogpt/{MHC_CONFIG} \
  --device "{DEVICE}" \
  --dtype "{DTYPE}" \
  --prompt "The future of machine learning is" \
  --max-new-tokens 80 \
  --temperature 0.8 \
  --top-k 50


/content/mhc
Generated text:
The future of machine learning is more.
There is a great business?
- It is a new for a beautiful and how it is a real estate or even more important way to ensure that the only good way they are not for your business for your company.
So here is an individual who is an important thing that they are very successful.
You could have a great book to do with an opportunity to get it.

Metrics:
device: cuda
dtype: float16
parameter_count: 11489872
prompt_length: 6
generated_tokens: 80
elapsed_seconds: 2.636848
tokens_per_sec: 30.339
peak_cuda_memory: 46.45 MB


## 12. Visualization: report-ready charts

Charts are saved to `{DRIVE_REPORTS}/figures`:
- `train_loss_curve.png`
- `val_loss_curve.png`
- `best_val_loss_bar.png`
- `final_val_ppl_bar.png`
- `tokens_per_sec_curve.png`
- `peak_vram_curve.png`
- `inference_tokens_per_sec_bar.png`
- `decode_ms_per_token_bar.png`
- `benchmark_peak_vram_bar.png`


In [14]:
%cd /content/mhc

if not PLOT_SCRIPT.exists():
    found = _find_repo_file("plot_training_comparison.py")
    if found is not None:
        PLOT_SCRIPT = found
    else:
        raise FileNotFoundError(
            "plot_training_comparison.py missing. Pull latest repo or copy examples/nanogpt/plot_training_comparison.py."
        )

FIG_DIR = DRIVE_REPORTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        "python", str(PLOT_SCRIPT),
        "--runs",
        f"baseline=examples/nanogpt/{BASELINE_OUT}",
        f"hc=examples/nanogpt/{HC_OUT}",
        f"mhc=examples/nanogpt/{MHC_OUT}",
        "--benchmark-summary", str(DRIVE_REPORTS / "benchmarks" / "summary.csv"),
        "--output-dir", str(FIG_DIR),
    ],
    cwd=str(REPO),
    check=True,
)

!ls -lh "{FIG_DIR}"


/content/mhc
total 511K
-rw------- 1 root root  22K May 20 14:38 benchmark_peak_vram_bar.png
-rw------- 1 root root  18K May 20 14:37 best_val_loss_bar.png
-rw------- 1 root root  20K May 20 14:38 decode_ms_per_token_bar.png
-rw------- 1 root root  23K May 20 14:37 final_val_ppl_bar.png
-rw------- 1 root root  24K May 20 14:37 inference_tokens_per_sec_bar.png
-rw------- 1 root root  39K May 20 14:37 peak_vram_curve.png
-rw------- 1 root root 145K May 20 14:37 tokens_per_sec_curve.png
-rw------- 1 root root  39K May 20 14:37 training_peak_vram_curve.png
-rw------- 1 root root 126K May 20 14:37 train_loss_curve.png
-rw------- 1 root root  57K May 20 14:37 val_loss_curve.png


## 13. How to report this experiment

Ready-to-say paragraph in Vietnamese:

> Do giới hạn tài nguyên Colab T4, em thiết kế một controlled small-scale experiment với cấu hình Mini. Ba biến thể baseline, HC và mHC được train from scratch với cùng kiến trúc, cùng dataset shard, cùng optimizer, cùng effective batch và cùng số iteration. Em so sánh bằng training loss, validation loss, validation perplexity, inference latency, throughput và peak VRAM. Kết quả này không nhằm reproduce toàn bộ benchmark của paper, mà nhằm kiểm chứng implementation và so sánh thực nghiệm trong điều kiện tài nguyên giới hạn.

Can claim:
- real training from scratch
- fair small-scale comparison
- validation loss / perplexity comparison
- inference runtime comparison

Must not claim:
- full paper reproduction
- MMLU/BBH/GSM8K results
- large-scale language model quality


## 14. Final artifact checklist

This verifies required files, summary status, same training budget, and expected residual flags.


In [15]:
from pathlib import Path
import pandas as pd
import json

required_paths = [
    NANOGPT / BASELINE_OUT / "ckpt.pt",
    NANOGPT / HC_OUT / "ckpt.pt",
    NANOGPT / MHC_OUT / "ckpt.pt",
    DRIVE_REPORTS / "training_summary.csv",
    DRIVE_REPORTS / "training_summary.md",
    DRIVE_REPORTS / "training_summary.json",
    DRIVE_REPORTS / "benchmarks" / "summary.csv",
    DRIVE_REPORTS / "benchmarks" / "summary.md",
    DRIVE_REPORTS / "figures" / "train_loss_curve.png",
    DRIVE_REPORTS / "figures" / "val_loss_curve.png",
    DRIVE_REPORTS / "figures" / "best_val_loss_bar.png",
    DRIVE_REPORTS / "figures" / "final_val_ppl_bar.png",
    DRIVE_REPORTS / "figures" / "tokens_per_sec_curve.png",
    DRIVE_REPORTS / "figures" / "peak_vram_curve.png",
    DRIVE_REPORTS / "figures" / "inference_tokens_per_sec_bar.png",
    DRIVE_REPORTS / "figures" / "decode_ms_per_token_bar.png",
    DRIVE_REPORTS / "figures" / "benchmark_peak_vram_bar.png",
]

missing = []
for path in required_paths:
    ok = path.exists()
    print(("OK " if ok else "MISS"), path)
    if not ok:
        missing.append(str(path))
if missing:
    raise FileNotFoundError("missing required artifacts: " + ", ".join(missing))

summary = pd.read_csv(DRIVE_REPORTS / "training_summary.csv")
print(summary[["variant", "ok", "best_val_loss", "final_val_loss", "final_val_ppl"]])
if not summary["ok"].astype(str).eq("True").all():
    raise AssertionError("not all summary rows have ok=True")

same_keys = [
    "max_iters",
    "batch_size",
    "gradient_accumulation_steps",
    "n_layer",
    "n_head",
    "n_embd",
    "block_size",
]
for key in same_keys:
    values = set(summary[key].astype(str))
    print(key, values)
    if len(values) != 1:
        raise AssertionError(f"variants do not share same {key}: {values}")

expected_flags = {
    "baseline": {"hc_disable": "True", "mhc": "False", "hc_num_streams": "1"},
    "hc": {"hc_disable": "False", "mhc": "False", "hc_num_streams": "4"},
    "mhc": {"hc_disable": "False", "mhc": "True", "hc_num_streams": "4"},
}
for _, row in summary.iterrows():
    variant = row["variant"]
    for key, expected in expected_flags[variant].items():
        actual = str(row[key])
        print(variant, key, actual)
        if actual != expected:
            raise AssertionError(f"{variant} {key}: {actual} != {expected}")

print("FINAL CHECKLIST PASSED")
print("DRIVE_REPORTS:", DRIVE_REPORTS)
print("DRIVE_RUNS:", DRIVE_RUNS)


OK  /content/mhc/examples/nanogpt/out-t4-mini-baseline/ckpt.pt
OK  /content/mhc/examples/nanogpt/out-t4-mini-hc/ckpt.pt
OK  /content/mhc/examples/nanogpt/out-t4-mini-mhc/ckpt.pt
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/training_summary.csv
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/training_summary.md
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/training_summary.json
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/benchmarks/summary.csv
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/benchmarks/summary.md
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/figures/train_loss_curve.png
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/figures/val_loss_curve.png
OK  /content/drive/MyDrive/mhc/reports/t4-mini-fineweb1shards-1500iters/figures/best_val_loss_bar.png
OK  /content/drive/MyDrive/mhc/reports/t4-mini-finew

## 15. Optional Full T4 Experiment — not recommended for Colab Free

Old full-ish workflow used `block_size=1024`, `n_layer=6`, `n_head=6`, `n_embd=288`, `batch_size=8`, `gradient_accumulation_steps=8`, and thousands of iterations. On Colab T4 this can take many hours per variant.

This optional path is disabled by default. Use only if you have enough GPU time and still keep all variants on same budget.


In [16]:
RUN_OPTIONAL_FULL = False

if RUN_OPTIONAL_FULL:
    run_cmd_live(
        [
            "bash", "examples/nanogpt/run_t4_full_compare.sh",
        ],
        cwd=REPO,
        log_path=LOG_DIR / "optional-full-t4.log",
    )
else:
    print("Optional full T4 experiment disabled. Mini is main report workflow.")


Optional full T4 experiment disabled. Mini is main report workflow.
